In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
paramaggarwal_fashion_product_images_dataset_path = kagglehub.dataset_download('paramaggarwal/fashion-product-images-dataset')

print('Data source import complete.')


# StyleFinder (AI Powered Fashion Analysis & Recommendation)

Multimodal RAG pipeline using the **Fashion Product Images Dataset**:



| Component | Implementation |
|-----------|----------------|
| Dataset | Fashion Product Images (Kaggle) |
| Image embedding | CLIP |
| Vector search | FAISS |
| Vision LLM | Qwen2-VL-2B-Instruct |
| Shopping links | Amazon, ASOS, Zara |
| Interface | Gradio  |




## 1) Install Dependencies


In [ ]:
%%bash
pip install -q \
    transformers accelerate bitsandbytes \
    faiss-cpu \
    gradio \
    pandas pillow \
    qwen-vl-utils \
    torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 48.9 MB/s eta 0:00:00


##  2) Configuration


In [ ]:
import os
import torch
from transformers.utils import logging as hf_logging

# Suppress verbose HuggingFace HTTP logs
hf_logging.set_verbosity_error()

# Dataset
DATASET_BASE  = '/kaggle/input/datasets/paramaggarwal/fashion-product-images-dataset'
STYLES_CSV    = os.path.join(DATASET_BASE, 'fashion-dataset','styles.csv')
IMAGES_DIR    = os.path.join(DATASET_BASE, 'fashion-dataset', 'images')

# Models
CLIP_MODEL_ID = 'openai/clip-vit-base-patch32'
QWEN_MODEL_ID = 'Qwen/Qwen2-VL-2B-Instruct'

# Search
TOP_K = 5

# Device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Configuration loaded.')
print(f'  CLIP  : {CLIP_MODEL_ID}')
print(f'  LLM   : {QWEN_MODEL_ID}')
print(f'  Device: {DEVICE}')
print(f'  Styles: {STYLES_CSV}')
print(f'  Images: {IMAGES_DIR}')


Configuration loaded.
  CLIP  : openai/clip-vit-base-patch32
  LLM   : Qwen/Qwen2-VL-2B-Instruct
  Device: cuda
  Styles: /kaggle/input/datasets/paramaggarwal/fashion-product-images-dataset/fashion-dataset/styles.csv
  Images: /kaggle/input/datasets/paramaggarwal/fashion-product-images-dataset/fashion-dataset/images


---

## 3) DatasetLoader




In [ ]:
import pandas as pd
import os


class DatasetLoader:

    def __init__(self, styles_csv=STYLES_CSV, images_dir=IMAGES_DIR, max_items=None):
        self.images_dir = images_dir
        self.df = self._load(styles_csv, max_items)

    def _load(self, csv_path, max_items):
        print(f'Loading styles from: {csv_path}')
        df = pd.read_csv(csv_path, on_bad_lines='skip')
        df['image_path'] = df['id'].apply(
            lambda x: os.path.join(self.images_dir, f'{x}.jpg')
        )
        # Keep only rows with existing images
        df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)
        if max_items:
            df = df.head(max_items)
        print(f'Dataset ready: {len(df)} products with images')
        return df

    def get_item(self, idx):
        """Return a single row as a dict."""
        row = self.df.iloc[idx]
        return {
            'id':                 row['id'],
            'productDisplayName': row.get('productDisplayName', ''),
            'masterCategory':     row.get('masterCategory', ''),
            'subCategory':        row.get('subCategory', ''),
            'articleType':        row.get('articleType', ''),
            'baseColour':         row.get('baseColour', ''),
            'season':             row.get('season', ''),
            'usage':              row.get('usage', ''),
            'image_path':         row['image_path'],
        }


print('DatasetLoader class defined.')


DatasetLoader class defined.


---
## 4) ImageEmbedder (CLIP)



In [ ]:
import torch
import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from tqdm.auto import tqdm


class ImageEmbedder:

    def __init__(self, model_id=CLIP_MODEL_ID, device=DEVICE):
        print(f'Loading CLIP: {model_id}...')
        self.device    = device
        self.processor = CLIPProcessor.from_pretrained(model_id)
        self.model     = CLIPModel.from_pretrained(model_id).to(device)
        self.model.eval()
        print(f'CLIP ready on {device}.')

    def _to_tensor(self, features):
        if isinstance(features, torch.Tensor):
            return features
        if hasattr(features, 'image_embeds') and features.image_embeds is not None:
            return features.image_embeds
        if hasattr(features, 'last_hidden_state'):
            return features.last_hidden_state[:, 0]
        for attr in vars(features).values():
            if isinstance(attr, torch.Tensor):
                return attr
        raise AttributeError(f'Cannot extract tensor from: {type(features)}')

    def _normalize(self, vecs):
        """L2-normalize rows with safe clipping to avoid division by zero."""
        norms = np.linalg.norm(vecs, axis=1, keepdims=True)
        return vecs / np.clip(norms, 1e-10, None)

    def embed_image(self, image_input):
        if isinstance(image_input, str):
            image = Image.open(image_input).convert('RGB')
        else:
            image = image_input.convert('RGB')

        inputs = self.processor(images=image, return_tensors='pt').to(self.device)
        with torch.no_grad():
            features = self.model.get_image_features(**inputs)
        vec = self._to_tensor(features).detach().cpu().numpy().astype('float32').flatten()
        norms = np.linalg.norm(vec)
        return vec / np.clip(norms, 1e-10, None)

    def embed_dataset(self, df, batch_size=64):
        all_vecs = []
        paths    = df['image_path'].tolist()
        print(f'Embedding {len(paths)} images (batch={batch_size})...')

        for i in tqdm(range(0, len(paths), batch_size), desc='Embedding'):
            batch_paths = paths[i:i+batch_size]
            images = []
            for p in batch_paths:
                try:
                    images.append(Image.open(p).convert('RGB'))
                except Exception:
                    images.append(Image.new('RGB', (224, 224)))

            inputs = self.processor(
                images=images, return_tensors='pt', padding=True
            ).to(self.device)
            with torch.no_grad():
                features = self.model.get_image_features(**inputs)

            vecs = self._to_tensor(features).detach().cpu().numpy().astype('float32')
            vecs = self._normalize(vecs)
            all_vecs.append(vecs)

        matrix = np.vstack(all_vecs)
        print(f'Embedding matrix shape: {matrix.shape}')
        return matrix


print('ImageEmbedder class defined.')

ImageEmbedder class defined.


---
##  5) VectorSearch (FAISS)


In [ ]:
import faiss
import numpy as np


class VectorSearch:
    """
    FAISS-based similarity search over CLIP embeddings.
    """

    def __init__(self):
        self.index = None
        self.df    = None

    def build_index(self, embeddings, df):
        """
        Build FAISS index from embedding matrix.
        Args:
            embeddings: np.ndarray shape (N, 512)
            df: DataFrame aligned with embeddings (same row order)
        """
        dim         = embeddings.shape[1]
        self.index  = faiss.IndexFlatIP(dim)  # inner product on normalized vecs = cosine sim
        self.index.add(embeddings)
        self.df     = df.reset_index(drop=True)
        print(f'FAISS index built: {self.index.ntotal} vectors, dim={dim}')

    def search(self, query_vec, top_k=TOP_K):
        """
        Search for top-K most similar products.
        Args:
            query_vec: np.ndarray shape (512,)
            top_k: number of results
        Returns:
            list of dicts: {productDisplayName, articleType, baseColour,
                            similarity, image_path, id}
        """
        if self.index is None:
            raise RuntimeError('Index not built. Call build_index() first.')

        q = query_vec.reshape(1, -1).astype('float32')
        scores, indices = self.index.search(q, top_k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < 0:
                continue
            row = self.df.iloc[idx]
            results.append({
                'productDisplayName': row.get('productDisplayName', 'Unknown'),
                'articleType':        row.get('articleType', ''),
                'baseColour':         row.get('baseColour', ''),
                'masterCategory':     row.get('masterCategory', ''),
                'similarity':         float(score),
                'image_path':         row['image_path'],
                'id':                 row['id'],
            })
        return results


print('VectorSearch class defined.')


VectorSearch class defined.


---
## 6) FashionAnalyzer (Qwen2-VL-2B)





In [ ]:
import base64, json, re, logging, os, torch
from io import BytesIO
from PIL import Image
from transformers import (
    Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
)
from transformers.utils import logging as hf_logging
from qwen_vl_utils import process_vision_info

hf_logging.set_verbosity_error()
logger = logging.getLogger(__name__)

FASHION_PROMPT = """
You are an expert fashion stylist AI that analyzes outfits from images.

Carefully examine the image and identify all visible clothing items and accessories worn by the person.

Return ONLY JSON in this exact format:

{
  "overview": "...",
  "items": [
    {"name": "...", "color": "..."}
  ],
  "style": "...",
  "styling_tips": ["...", "..."]
}

Rules:
- Detect every visible garment and accessory (jackets, shirts, dresses, trousers, skirts, scarves, hijabs, sunglasses, bags, shoes, belts, hats).
- Determine the dominant color for each item.
- List each item separately.
- style field: overall fashion aesthetic (casual, streetwear, minimalist, formal, modest fashion, sporty, chic, etc.)
- styling_tips: short and practical.
- Do NOT invent items not visible in the image.
- Do NOT include brand names, prices, or product recommendations.
- items list must NOT be empty if any clothing is visible.
- Output valid JSON ONLY. No explanations, no markdown fences.
"""


class FashionAnalyzer:

    def __init__(self, model_id=QWEN_MODEL_ID, hf_token=None,
                 temperature=0.4, max_new_tokens=1028):
        print(f'Loading {model_id} (4-bit NF4)...')
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4',
        )
        token = hf_token or os.environ.get('HF_TOKEN') or None
        self.processor = AutoProcessor.from_pretrained(model_id, token=token)
        self.model = Qwen2VLForConditionalGeneration.from_pretrained(
            model_id,
            quantization_config=bnb_cfg,
            device_map='auto',
            torch_dtype=torch.float16,
            token=token,
        )
        self.model.eval()
        self.temperature    = temperature
        self.max_new_tokens = max_new_tokens
        print('FashionAnalyzer ready.')

    def _infer(self, image, prompt):
        messages = [{'role': 'user', 'content': [
            {'type': 'image', 'image': image},
            {'type': 'text',  'text': prompt},
        ]}]
        text_input = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        img_inputs, vid_inputs = process_vision_info(messages)
        inputs = self.processor(
            text=[text_input], images=img_inputs,
            videos=vid_inputs, padding=True, return_tensors='pt',
        ).to(self.model.device)

        with torch.no_grad():
            out_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=self.temperature > 0,
                temperature=self.temperature if self.temperature > 0 else None,
                repetition_penalty=1.1,
            )
        trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out_ids)]
        return self.processor.batch_decode(
            trimmed, skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )[0].strip()

    def analyze(self, image_input):
        # Normalize to PIL Image
        if isinstance(image_input, str):
            if os.path.exists(image_input):
                image = Image.open(image_input).convert('RGB')
            else:
                img_bytes = base64.b64decode(image_input)
                image = Image.open(BytesIO(img_bytes)).convert('RGB')
        else:
            image = image_input.convert('RGB')

        raw = self._infer(image, FASHION_PROMPT)

        parsed = None

        # Attempt 1: strip markdown fences and parse directly
        try:
            clean  = re.sub(r'```(?:json)?|```', '', raw).strip()
            parsed = json.loads(clean)
        except Exception:
            pass

        # Attempt 2: extract first {...} block
        if parsed is None:
            try:
                match = re.search(r'\{[\s\S]*\}', raw)
                if match:
                    parsed = json.loads(match.group(0))
            except Exception:
                pass

        # Attempt 3: double-encoded JSON inside overview field
        if parsed is None:
            try:
                inner = re.search(r'"overview"\s*:\s*"(\{[\s\S]*?\})"', raw)
                if inner:
                    parsed = json.loads(inner.group(1).replace('\\"', '"'))
            except Exception:
                pass

        # Fallback
        if parsed is None:
            parsed = {
                'overview':     raw[:300] if raw else 'Analysis unavailable.',
                'items':        [],
                'style':        'unknown',
                'styling_tips': []
            }

        # Ensure overview is a clean string
        overview = parsed.get('overview', '')
        if not isinstance(overview, str):
            overview = str(overview)
        if overview.strip().startswith('{'):
            overview = 'Outfit analysis complete. See detected items below.'

        items = parsed.get('items', [])
        if not isinstance(items, list):
            items = []
        items = [i for i in items if isinstance(i, dict) and i.get('name')]

        styling_tips = parsed.get('styling_tips', [])
        if not isinstance(styling_tips, list):
            styling_tips = []

        return {
            'overview':     overview,
            'items':        items,
            'style':        parsed.get('style', 'unknown'),
            'styling_tips': styling_tips,
        }


print('FashionAnalyzer class defined.')

FashionAnalyzer class defined.


---
## 7) LinkGenerator



| Store | URL Template |
|-------|-------------|
| Amazon | `https://www.amazon.com/s?k={query}` |
| ASOS | `https://www.asos.com/search/?q={query}` |
| Zara | `https://www.zara.com/us/en/search?searchTerm={query}` |


In [ ]:
import urllib.parse


class LinkGenerator:
    """
    Builds real shopping search links from detected clothing item names.
    No hallucinated product names, brands, or prices.
    """

    STORES = {
        'Amazon': 'https://www.amazon.com/s?k={query}',
        'ASOS':   'https://www.asos.com/search/?q={query}',
        'Zara':   'https://www.zara.com/us/en/search?searchTerm={query}',
    }

    def build_links(self, item_name, color=''):
        """Return {store: url} dict for one item."""
        query = f'{color} {item_name}'.strip() if color else item_name
        q_enc = urllib.parse.quote_plus(query)
        return {store: tpl.format(query=q_enc) for store, tpl in self.STORES.items()}

    def format_markdown(self, items):
        """Build Markdown shopping section from [{name, color}] list."""
        if not items:
            return '_No items detected._'
        lines = []
        for item in items:
            name  = item.get('name', '')
            color = item.get('color', '')
            if not name:
                continue
            links = self.build_links(name, color)
            parts = '  |  '.join(f'[{s}]({u})' for s, u in links.items())
            lines.append(f'- **{name.title()}** -- {parts}')
        return '\n'.join(lines)


print('LinkGenerator class defined.')


LinkGenerator class defined.


---
## 8) StyleFinderApp (Pipeline Orchestrator)


In [ ]:
import os, logging
from PIL import Image

logger = logging.getLogger(__name__)

# Map detected item names to dataset articleTypes for guided search
ITEM_TYPE_MAP = {
    'jacket': 'Jackets',   'blazer': 'Blazers',   'coat': 'Coats',
    'top': 'Tops',         'shirt': 'Shirts',      'blouse': 'Tops',
    'dress': 'Dresses',    'gown': 'Dresses',      'kurta': 'Kurtas',
    'jeans': 'Jeans',      'trousers': 'Trousers', 'skirt': 'Skirts',
    'shorts': 'Shorts',    'leggings': 'Leggings', 'sweater': 'Sweaters',
    'hijab': 'Scarves',    'scarf': 'Scarves',     'dupatta': 'Dupatta',
    'sunglasses': 'Sunglasses',                     'bag': 'Handbags',
    'handbag': 'Handbags', 'clutch': 'Clutches',
    'sandals': 'Sandals',  'heels': 'Heels',       'shoes': 'Flats',
}


class StyleFinderApp:

    def __init__(self, dataset_loader, image_embedder, vector_search,
                 fashion_analyzer, link_generator):
        self.loader   = dataset_loader
        self.embedder = image_embedder
        self.search   = vector_search
        self.analyzer = fashion_analyzer
        self.linker   = link_generator

    def _guided_search(self, query_vec, detected_items, top_k=TOP_K):
        """
        Search FAISS but re-rank by filtering to detected item types first.
        Falls back to global search if no type match found.
        """
        # Get detected article types from item names
        detected_types = set()
        for item in detected_items:
            name = item.get('name', '').lower()
            for keyword, article_type in ITEM_TYPE_MAP.items():
                if keyword in name:
                    detected_types.add(article_type)

        if detected_types:
            # Filter dataset to matching types and rebuild a temporary search
            filtered_df = self.loader.df[
                self.loader.df['articleType'].isin(detected_types)
            ].reset_index(drop=True)

            if len(filtered_df) >= top_k:
                # Get indices of filtered rows in the original df
                filtered_indices = filtered_df.index.tolist()
                import faiss, numpy as np
                sub_embeddings = np.vstack([
                    self.search.index.reconstruct(int(i))
                    for i in filtered_indices
                ])
                sub_index = faiss.IndexFlatIP(sub_embeddings.shape[1])
                sub_index.add(sub_embeddings)
                scores, idxs = sub_index.search(
                    query_vec.reshape(1, -1).astype('float32'), top_k
                )
                results = []
                for score, idx in zip(scores[0], idxs[0]):
                    if idx < 0:
                        continue
                    row = filtered_df.iloc[idx]
                    results.append({
                        'productDisplayName': row.get('productDisplayName', ''),
                        'articleType':        row.get('articleType', ''),
                        'baseColour':         row.get('baseColour', ''),
                        'masterCategory':     row.get('masterCategory', ''),
                        'similarity':         float(score),
                        'image_path':         row['image_path'],
                        'id':                 row['id'],
                    })
                return results

        # Fallback: global search
        return self.search.search(query_vec, top_k=top_k)

    def process_image(self, image):
        try:
            if isinstance(image, str):
                pil_image = Image.open(image).convert('RGB')
            else:
                pil_image = image.convert('RGB')

            # Step 1: CLIP embedding
            query_vec = self.embedder.embed_image(pil_image)

            # Step 2: Qwen2-VL analysis first (needed for guided search)
            raw_analysis = self.analyzer.analyze(pil_image)

            if isinstance(raw_analysis, str):
                analysis = {'overview': raw_analysis, 'items': [],
                            'style': 'unknown', 'styling_tips': []}
            elif isinstance(raw_analysis, dict):
                analysis = raw_analysis
            else:
                analysis = {'overview': str(raw_analysis), 'items': [],
                            'style': 'unknown', 'styling_tips': []}

            overview     = analysis.get('overview', '')
            items        = analysis.get('items', [])
            style        = analysis.get('style', 'unknown')
            styling_tips = analysis.get('styling_tips', [])

            if not isinstance(items, list):
                items = []
            items = [i for i in items if isinstance(i, dict) and i.get('name')]

            # Step 3: Item-guided FAISS search
            results = self._guided_search(query_vec, items, top_k=TOP_K)

            # Step 4: Shopping links
            shopping_md = self.linker.format_markdown(items)

            # Step 5: Format output
            items_md = '\n'.join(
                f'- **{i.get("name","").title()}** ({i.get("color","")})'
                for i in items
            ) or '_No items detected_'

            tips_md = '\n'.join(
                f'{n+1}. {t}' for n, t in enumerate(styling_tips)
                if isinstance(t, str)
            ) or '_No tips generated_'

            similar_md_lines = []
            for r in results:
                sim_pct = f"{r['similarity']*100:.1f}%"
                similar_md_lines.append(
                    f'- **{r["productDisplayName"]}**  '
                    f'({r["articleType"]} | {r["baseColour"]} | {sim_pct} match)'
                )
            similar_md = '\n'.join(similar_md_lines) or '_No similar products found_'

            output = (
                '## Outfit Overview\n'
                f'{overview or "_No overview generated._"}\n'
                '\n---\n'
                '\n## Detected Items\n'
                f'{items_md}\n'
                f'\n**Style Classification:** {style.title()}\n'
                '\n---\n'
                '\n## Shop This Look\n'
                '_Search links built from detected items only._\n'
                f'\n{shopping_md}\n'
                '\n---\n'
                '\n## Similar Products From Dataset\n'
                f'{similar_md}\n'
            )

            # Step 6: Gallery
            gallery = []
            for r in results:
                try:
                    img     = Image.open(r['image_path']).convert('RGB')
                    sim_pct = f"{r['similarity']*100:.1f}%"
                    caption = (
                        f"{r['productDisplayName']}\n"
                        f"{r['articleType']} | {r['baseColour']} | {sim_pct}"
                    )
                    gallery.append((img, caption))
                except Exception as e:
                    logger.warning('Could not load image %s: %s', r['image_path'], e)

            return output.strip(), gallery

        except Exception as exc:
            logger.error('process_image error: %s', exc)
            return f'An error occurred: {exc}', []


print('StyleFinderApp class defined.')

StyleFinderApp class defined.


---
## 9) Gradio Interface




In [ ]:
import gradio as gr

CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Cormorant+Garamond:wght@300;400;600&family=Jost:wght@300;400;500&display=swap');

:root {
    --bg:      #08080d;
    --surface: #0f0f16;
    --border:  #1e1e2e;
    --gold:    #c9a96e;
    --rose:    #c96e8a;
    --teal:    #6ec9b8;
    --text:    #ede9e1;
    --muted:   #5a5a7a;
}

*, *::before, *::after { box-sizing: border-box; }

body, .gradio-container {
    background: var(--bg) !important;
    font-family: 'Jost', sans-serif !important;
    color: var(--text) !important;
}

.sf-header {
    text-align: center;
    padding: 3.5rem 1rem 2rem;
    border-bottom: 1px solid var(--border);
    margin-bottom: 2rem;
    background: radial-gradient(ellipse at 50% -20%, rgba(201,169,110,0.06) 0%, transparent 65%);
}
.sf-header h1 {
    font-family: 'Cormorant Garamond', serif;
    font-size: 3.8rem;
    font-weight: 300;
    letter-spacing: 0.22em;
    text-transform: uppercase;
    background: linear-gradient(120deg, #c9a96e, #c96e8a, #6ec9b8);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
    margin: 0 0 0.5rem;
}
.sf-header p {
    font-size: 0.72rem;
    letter-spacing: 0.4em;
    text-transform: uppercase;
    color: var(--muted);
    margin: 0;
}
.sf-label {
    font-size: 0.62rem;
    letter-spacing: 0.35em;
    text-transform: uppercase;
    color: var(--muted);
    margin-bottom: 0.6rem;
}
.sf-status {
    font-size: 0.68rem;
    letter-spacing: 0.2em;
    text-transform: uppercase;
    color: var(--teal);
    text-align: center;
    padding: 0.5rem;
    border: 1px solid var(--border);
    border-radius: 6px;
    margin-top: 0.6rem;
    background: var(--surface);
}
.btn-analyze {
    background: linear-gradient(135deg, #b8924a, #c96e8a) !important;
    border: none !important;
    border-radius: 8px !important;
    color: #fff !important;
    font-size: 0.78rem !important;
    letter-spacing: 0.22em !important;
    text-transform: uppercase !important;
    padding: 0.8rem !important;
    width: 100% !important;
    margin-top: 0.6rem !important;
    box-shadow: 0 4px 20px rgba(201,169,110,0.18) !important;
    transition: opacity 0.2s, transform 0.15s !important;
}
.btn-analyze:hover { opacity: 0.85 !important; transform: translateY(-1px) !important; }

/* Output panel — target by elem_id */
#output-panel {
    background: var(--surface) !important;
    border: 1px solid var(--border) !important;
    border-radius: 12px !important;
    padding: 1.8rem !important;
    min-height: 500px !important;
    line-height: 1.85 !important;
    color: var(--text) !important;
}
#output-panel h2 {
    font-family: 'Cormorant Garamond', serif !important;
    font-weight: 400 !important;
    color: var(--gold) !important;
    border-bottom: 1px solid var(--border);
    padding-bottom: 0.25rem;
    margin-top: 1.2rem;
}
#output-panel h3  { color: var(--rose) !important; font-size: 0.9rem !important; }
#output-panel a   { color: var(--teal) !important; text-decoration: none !important; }
#output-panel li  { color: var(--muted); margin-bottom: 0.3rem; }
#output-panel strong { color: var(--text) !important; }

.sf-gallery { margin-top: 0.5rem; }
.sf-footer {
    text-align: center;
    padding: 1.2rem;
    color: var(--muted);
    font-size: 0.62rem;
    letter-spacing: 0.25em;
    text-transform: uppercase;
    border-top: 1px solid var(--border);
    margin-top: 2rem;
}
"""

PLACEHOLDER = """\
*Upload an outfit photo and click **Analyze Style** to begin.*

The report will include:

- Outfit overview
- Detected clothing items & colors
- Style classification
- Styling suggestions
- Shopping links (Amazon · ASOS · Zara)
- Similar products from the fashion dataset
"""


def create_gradio_interface(app):

    with gr.Blocks(
        css=CUSTOM_CSS,
        title="StyleFinder",
        theme=gr.themes.Base(
            primary_hue=gr.themes.colors.orange,
            neutral_hue=gr.themes.colors.slate,
        ),
    ) as demo:

        gr.HTML("""
        <div class="sf-header">
            <h1>Style Finder</h1>
            <p>AI-Powered Fashion Analysis &amp; Recommendations</p>
        </div>
        """)

        with gr.Row(equal_height=False):

            with gr.Column(scale=4, min_width=280):
                gr.HTML('<div class="sf-label">Your outfit</div>')
                image_input = gr.Image(
                    type="pil",
                    label="Upload image",
                    height=340,
                    sources=["upload", "clipboard"],
                )
                analyze_btn = gr.Button(
                    "Analyze Style",
                    variant="primary",
                    elem_classes="btn-analyze",
                )
                status = gr.HTML('<div class="sf-status">Ready</div>')

                with gr.Accordion("How it works", open=False):
                    gr.Markdown("""\
1. Image is encoded and matched to similar products
2. Vision model analyzes the outfit style
3. Shopping links are generated per detected item
""")
                with gr.Accordion("Tips", open=False):
                    gr.Markdown("""\
- Well-lit, full-body or upper-body photos work best
- Avoid heavily cropped or blurry images
- JPG or PNG recommended
""")

            with gr.Column(scale=6, min_width=380):
                gr.HTML('<div class="sf-label">Analysis report</div>')
                # Use elem_id instead of elem_classes for reliable CSS targeting
                output = gr.Markdown(
                    value=PLACEHOLDER,
                    elem_id="output-panel",
                )

        gr.HTML('<div class="sf-label" style="margin-top:1.8rem">Similar products</div>')
        gallery = gr.Gallery(
            label="",
            show_label=False,
            elem_classes="sf-gallery",
            columns=5,
            rows=1,
            height=270,
            object_fit="cover",
        )

        gr.HTML("""
        <div class="sf-footer">
            StyleFinder &nbsp;&middot;&nbsp; Fashion Analysis &nbsp;&middot;&nbsp; Powered by CLIP &amp; Qwen2-VL
        </div>
        """)

        def analyzing():
            return '<div class="sf-status" style="color:#c9a96e">Analyzing your image...</div>'

        def done():
            return '<div class="sf-status" style="color:#6ec9b8">Analysis complete</div>'

        analyze_btn.click(
            fn=analyzing, inputs=None, outputs=status
        ).then(
            fn=app.process_image,
            inputs=image_input,
            outputs=[output, gallery],
        ).then(
            fn=done, inputs=None, outputs=status
        )

    return demo


print("create_gradio_interface() defined.")

create_gradio_interface() defined.


---
## 10) Initialize Components & Build FAISS Index

This cell:
1. Loads the dataset
2. Loads CLIP and embeds all dataset images
3. Builds the FAISS index
4. Loads Qwen2-VL-2B



In [ ]:
import os
import numpy as np

MAX_ITEMS  = None
EMBED_FILE = "fashion_embeddings.npy"

# Step 1: Load full dataset
loader = DatasetLoader(
    styles_csv=STYLES_CSV,
    images_dir=IMAGES_DIR,
    max_items=MAX_ITEMS,
)
print(f"Dataset loaded: {len(loader.df)} items")

# Step 2: CLIP embeddings
embedder = ImageEmbedder(model_id=CLIP_MODEL_ID, device=DEVICE)

if os.path.exists(EMBED_FILE):
    embeddings = np.load(EMBED_FILE)
    print(f"Cached embeddings shape: {embeddings.shape}")

    # Validate cache matches current dataset
    if len(embeddings) != len(loader.df):
        print(f"Cache mismatch ({len(embeddings)} vs {len(loader.df)}) — regenerating...")
        os.remove(EMBED_FILE)
        embeddings = None
    else:
        print("Cache valid, skipping embedding generation.")

if not os.path.exists(EMBED_FILE):
    print(f"Generating CLIP embeddings for {len(loader.df)} images (~15 min)...")
    embeddings = embedder.embed_dataset(loader.df, batch_size=128)
    np.save(EMBED_FILE, embeddings)
    print(f"Embeddings saved to {EMBED_FILE}")

print(f"Embedding shape: {embeddings.shape}")

# Step 3: Build FAISS index
searcher = VectorSearch()
searcher.build_index(embeddings, loader.df)
print("FAISS index ready.")

# Step 4: Load Qwen2-VL analyzer
HF_TOKEN = os.environ.get("HF_TOKEN", "") or None
analyzer = FashionAnalyzer(model_id=QWEN_MODEL_ID, hf_token=HF_TOKEN)

# Step 5: Link generator
linker = LinkGenerator()

print("All components initialized successfully.")

Loading styles from: /kaggle/input/datasets/paramaggarwal/fashion-product-images-dataset/fashion-dataset/styles.csv
Dataset ready: 44419 products with images
Dataset loaded: 44419 items
Loading CLIP: openai/clip-vit-base-patch32...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

FashionAnalyzer ready.
All components initialized successfully.


---
## 11) Launch Gradio App




In [ ]:
# Assemble the app
style_app = StyleFinderApp(
    dataset_loader  = loader,
    image_embedder  = embedder,
    vector_search   = searcher,
    fashion_analyzer= analyzer,
    link_generator  = linker,
)

# Build Gradio interface
demo = create_gradio_interface(style_app)

# Launch -- share=True gives a public URL for 72 hours
demo.launch(share=True, show_error=False)


/tmp/ipykernel_55/2638665732.py:138: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_55/2638665732.py:138: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7869
* Running on public URL: https://f91e46ecd8e03a2660.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1139, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error



---
### Upgrade Path

```
Current:  CLIP ViT-B/32 -> FAISS -> Qwen2-VL-2B

Upgrade next:  CLIP ViT-L/14 -> FAISS -> Qwen2-VL-7B  or something else

```

